In [1]:
%pip install pandas numpy matplotlib seaborn sqlalchemy "psycopg[binary]" plotly

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 6.8 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 10.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 15.9 MB/s  0:00:00 eta 0:00:01
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 18.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 20.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 25.8 MB/s  0:00:00 eta 0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 29.4 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl 

In [1]:
import sys

import matplotlib
import numpy as np
import pandas as pd
import plotly
import seaborn as sns
import sqlalchemy
import psycopg

print(f"Python: {sys.version}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {sns.__version__}")
print(f"SQLAlchemy: {sqlalchemy.__version__}")
print(f"psycopg: {psycopg.__version__}")
print(f"plotly: {plotly.__version__}")

Matplotlib is building the font cache; this may take a moment.


Python: 3.13.15 (v3.13.15:4061bc4c35f, Aug  5 2026, 09:08:51) [Clang 21.0.0 (clang-2100.1.1.101)]
pandas: 3.0.6
numpy: 2.5.3
matplotlib: 3.11.2
seaborn: 0.13.2
SQLAlchemy: 2.0.54
psycopg: 3.3.6
plotly: 7.1.0


In [2]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import os

from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

load_dotenv(ENV_PATH)

db_url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(db_url)

In [2]:
with engine.connect() as connection:
    postgres_version = connection.execute(
        text("SELECT version();")
    ).scalar()

print(postgres_version)

PostgreSQL 18.6 on aarch64-apple-darwin24.6.0, compiled by Apple clang version 17.0.0 (clang-1700.0.13.5), 64-bit


In [3]:
schema_tables_query = """
SELECT
    table_name
FROM information_schema.tables
WHERE table_schema = 'data_jobs'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;
"""

with engine.connect() as connection:
    schema_tables = connection.execute(
        text(schema_tables_query)
    ).fetchall()

for row in schema_tables:
    print(row[0])

company_dim
job_postings_fact
skills_dim
skills_job_dim


In [4]:
import pandas as pd

tables_to_load = [
    "company_dim",
    "job_postings_fact",
    "skills_dim",
    "skills_job_dim",
]

dataframes = {}

for table_name in tables_to_load:
    query = f"SELECT * FROM data_jobs.{table_name};"
    
    dataframes[table_name] = pd.read_sql_query(
        sql=text(query),
        con=engine,
    )

company_dim = dataframes["company_dim"]
job_postings_fact = dataframes["job_postings_fact"]
skills_dim = dataframes["skills_dim"]
skills_job_dim = dataframes["skills_job_dim"]

In [5]:
for table_name, dataframe in dataframes.items():
    rows, columns = dataframe.shape
    print(f"{table_name}: {rows:,} rows × {columns} columns")

company_dim: 98,372 rows × 5 columns
job_postings_fact: 478,895 rows × 16 columns
skills_dim: 254 rows × 3 columns
skills_job_dim: 2,274,756 rows × 2 columns


In [6]:
for table_name, dataframe in dataframes.items():
    print(f"\n{'=' * 70}")
    print(f"{table_name.upper()} — data types")
    display(dataframe.dtypes.to_frame(name="dtype"))
    
    print(f"\n{table_name.upper()} — first 3 rows")
    display(dataframe.head(3))


COMPANY_DIM — data types


,dtype
company_id,int64
name,str
link,str
link_google,str
thumbnail,str



COMPANY_DIM — first 3 rows


,company_id,name,link,link_google,thumbnail
0,0,BJ's Wholesale Club,http://www.bjs.com/,https://www.google.com/search?sca_esv=59484023...,https://encrypted-tbn0.gstatic.com/images?q=tb...
1,1,Tesla,http://www.tesla.com/,https://www.google.com/search?sca_esv=59484023...,https://encrypted-tbn0.gstatic.com/images?q=tb...
2,2,Next Recruiting,NaN,https://www.google.com/search?sca_esv=4e3dd394...,NaN



JOB_POSTINGS_FACT — data types


...

,skill_id,skills,type
0,0,sql,programming
1,1,python,programming
2,2,r,programming



SKILLS_JOB_DIM — data types


,dtype
job_id,int64
skill_id,int64



SKILLS_JOB_DIM — first 3 rows


,job_id,skill_id
0,5,0
1,5,1
2,5,2


In [7]:
key_validation = pd.DataFrame(
    {
        "table": [
            "company_dim",
            "job_postings_fact",
            "skills_dim",
            "skills_job_dim",
        ],
        "expected_key": [
            "company_id",
            "job_id",
            "skill_id",
            "(job_id, skill_id)",
        ],
        "row_count": [
            len(company_dim),
            len(job_postings_fact),
            len(skills_dim),
            len(skills_job_dim),
        ],
        "unique_key_count": [
            company_dim["company_id"].nunique(),
            job_postings_fact["job_id"].nunique(),
            skills_dim["skill_id"].nunique(),
            skills_job_dim.drop_duplicates(
                subset=["job_id", "skill_id"]
            ).shape[0],
        ],
    }
)

key_validation["duplicate_key_rows"] = (
    key_validation["row_count"] - key_validation["unique_key_count"]
)

display(key_validation)

,table,expected_key,row_count,unique_key_count,duplicate_key_rows
0,company_dim,company_id,98372,98372,0
1,job_postings_fact,job_id,478895,478895,0
2,skills_dim,skill_id,254,254,0
3,skills_job_dim,"(job_id, skill_id)",2274756,2274756,0
